In [1]:
import h5py
import torch
import numpy as np
import os
import json

import keyrank_rs

import torch.distributed as dist

In [2]:
def setup(rank, world_size):
    dist.init_process_group(
        backend="nccl",   #"gloo", # gloo, nccl causes error
        init_method="tcp://127.0.0.1:29500",
        rank=rank,
        world_size=world_size,
        device_id=None,
    )

setup(0, 1)

In [3]:
if os.name == "nt":
    DATAFOLDER = "C:/Data"
else:
    DATAFOLDER = "/mnt/c/Data"

val_test_hdf = h5py.File(f"{DATAFOLDER}/simpleserial-aes-fix-500-diff.hdf5")

val_test_traces = torch.Tensor(np.array(val_test_hdf['trace']))
val_test_plaintexts = torch.Tensor(np.array(val_test_hdf['data']))
val_test_keys = torch.Tensor(np.array(val_test_hdf['key']))

device = torch.device("cuda")

In [4]:
print(val_test_traces.shape)
print(val_test_plaintexts.shape)
print(val_test_keys.shape)

torch.Size([1000, 500, 5000])
torch.Size([1000, 500, 32])
torch.Size([1000, 16])


In [5]:
def metadata_best_epoch(model_name) -> int:
    with open(f"models/{model_name}/metadata.json") as f:
        metadata = json.load(f)
        val_scores = metadata["scores"][1]
        best_epoch = np.array(val_scores).argmin()
    return best_epoch.item()

def get_traces_mean_std(trace_start, trace_end):
    """Load the mean and std of the training trace set within the given interval"""
    with open(f"misc/standardization/trace{trace_start}_{trace_end}.json") as f:
        info = json.load(f)
        mean = info['training_traces_mean']
        std = info['training_traces_std']

    return mean, std

In [6]:
IMPL = "fixslice"
ARCH = "transnet"
PREDICTION_TARGET = "sbox"
TARGET_BYTE_IDX = 1
TRACE_START = 400
TRACE_END = 1500
SEED = 777

model_name = f"{IMPL}-{PREDICTION_TARGET}-byte{TARGET_BYTE_IDX}-{ARCH}-{TRACE_START}_{TRACE_END}-s{SEED}"

epoch = 199

model_path = f"models/{model_name}/epoch{epoch}.pt"
print(model_path)

model = torch.load(model_path, weights_only=False).to(device)

models/fixslice-sbox-byte1-transnet-400_1500-s777/epoch199.pt


In [7]:
sample = 566

traces_mean, traces_std = get_traces_mean_std(TRACE_START, TRACE_END)

traces = (val_test_traces[sample, :, TRACE_START:TRACE_END] - traces_mean) / traces_std
plaintexts = val_test_plaintexts[sample, :, :16] # first plaintext block
key = val_test_keys[sample]

print(traces.shape)
print(plaintexts.shape)
print(key.shape)

torch.Size([500, 1100])
torch.Size([500, 16])
torch.Size([16])


In [8]:
print("True key:", key.long().tolist())
true_key =  key.long().tolist()

sbox_scores = model(traces.to(device))
numpy_scores_ = sbox_scores.detach().cpu().numpy()

for n_traces in range(2,100):
    guesses = []

    numpy_scores = numpy_scores_[:n_traces]

    for idx in range(16):

        plaintext_bytes = plaintexts[:n_traces, idx]
        plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

        numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_scores)

        x = torch.Tensor(numpy_keyscores).softmax(dim=1)
        x = x.log()
        x = x.sum(dim=0)
        x = x.argmax(dim=0)

        guesses.append(x.item())

    if true_key == guesses:
        print(n_traces+1,"traces")
        break

print("Full attack:",guesses)    

True key: [50, 33, 207, 182, 225, 235, 22, 176, 124, 53, 45, 42, 254, 24, 91, 209]


OutOfMemoryError: CUDA out of memory. Tried to allocate 8.85 GiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Including non-PyTorch memory, this process has 17179869184.00 GiB memory in use. Of the allocated memory 15.63 GiB is allocated by PyTorch, and 19.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [10]:
true_key =  key.long().tolist()

n_traces = 20

guesses = []

sbox_scores = model(traces[:n_traces].to(device))
numpy_scores = sbox_scores.detach().cpu().numpy()

for idx in range(16):

    plaintext_bytes = plaintexts[:n_traces, idx]
    plaintext_bytes = plaintext_bytes.long().detach().cpu().numpy().squeeze()

    numpy_keyscores = keyrank_rs.sbox_scores_to_keyscores_parallel(plaintext_bytes, numpy_scores)

    x = torch.Tensor(numpy_keyscores).softmax(dim=1)
    x = x.log()
    x = x.sum(dim=0)
    x = x.argmax(dim=0)

    guesses.append(x.item())

print("True key:", true_key)
print("Full attack:",guesses)    

true_key = [f"b{b:08b}" for b in true_key]
guesses = [f"b{b:08b}" for b in guesses]

#print("True key:", true_key)
#print("Full attack:",guesses)    


True key: [50, 33, 207, 182, 225, 235, 22, 176, 124, 53, 45, 42, 254, 24, 91, 209]
Full attack: [50, 33, 207, 11, 225, 202, 22, 176, 27, 217, 45, 148, 254, 60, 211, 255]
